# 本地验证脚本（评分一致性）
该笔记本只负责计算评分：
- 混合评分 = 0.3 × C 指数 + 0.7 × (1 - 加权布里尔评分)
- 加权布里尔评分 = 0.3 × Brier@24 + 0.4 × Brier@48 + 0.3 × Brier@72

请先在模型笔记本中生成预测文件（例如 data_clean/preds_baseline.csv）。

In [ ]:
import pandas as pd
import numpy as np
from sksurv.metrics import concordance_index_censored, brier_score

# 1. 读取真实数据与预测结果
真实数据来自 data_clean/train_clean.csv，预测结果来自模型输出的 CSV。
预测文件建议包含：
- event_id（可选，如果没有则按行对齐）
- pred_12, pred_24, pred_48, pred_72（生存概率 S(t)）
- risk_score（可选，用于 C-index；若没有则使用 1 - pred_48）

In [ ]:
# 路径可根据需要调整
train_path = "data_clean/train_clean.csv"
pred_path = "data_clean/preds_baseline.csv"

# 读取真实数据
truth = pd.read_csv(train_path)

# 读取预测结果
preds = pd.read_csv(pred_path)

# 对齐数据：优先按 event_id 匹配，否则按行号对齐
if "event_id" in truth.columns and "event_id" in preds.columns:
    df = truth.merge(preds, on="event_id", how="inner")
else:
    df = truth.copy()
    preds = preds.reset_index(drop=True)
    df = df.reset_index(drop=True)
    for col in preds.columns:
        if col not in df.columns:
            df[col] = preds[col]

# 必要字段检查
required_cols = ["event", "time_to_hit_hours", "pred_12", "pred_24", "pred_48", "pred_72"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"缺少必要字段: {missing}")

# 2. 指标函数
- C-index 使用风险排序（默认用 risk_score，如没有则用 1 - pred_48）
- Brier Score 使用生存概率 S(t)

In [ ]:
# 构造生存分析所需的结构化数组

def make_y(event_series, duration_series):
    return np.array(
        list(zip(event_series.astype(bool), duration_series)),
        dtype=[("event", "?"), ("duration", "<f8")],
    )


def calculate_c_index(y_true, risk_scores):
    return concordance_index_censored(y_true["event"], y_true["duration"], risk_scores)[0]


def calculate_weighted_brier(y_train, y_val, surv_probs, times=(24, 48, 72)):
    weights = [0.3, 0.4, 0.3]
    _, scores = brier_score(y_train, y_val, surv_probs, times)
    return weights[0] * scores[0] + weights[1] * scores[1] + weights[2] * scores[2]


def calculate_mixed_score(c_index, weighted_brier):
    return 0.3 * c_index + 0.7 * (1 - weighted_brier)

# 3. 计算本地评分

In [ ]:
# 组装 y
# 注意：duration 字段仍然命名为 "duration" 只是内部结构，外部列仍是 time_to_hit_hours

# 为了把随访范围扩展到 72 小时，追加一个“仅用于评估”的右删失样本
# 注意：brier_score 要求 times < max_follow_up，因此把伪样本时间设为 72.0001

eval_df = df.copy()
pseudo = {
    "event": 0,
    "time_to_hit_hours": 72.0001,
    "pred_12": 1.0,
    "pred_24": 1.0,
    "pred_48": 1.0,
    "pred_72": 1.0,
    "risk_score": 0.0,
}
for key in pseudo:
    if key not in eval_df.columns:
        eval_df[key] = np.nan

eval_df = pd.concat([eval_df, pd.DataFrame([pseudo])], ignore_index=True)

y = make_y(eval_df["event"], eval_df["time_to_hit_hours"])

# 风险评分：优先使用模型输出的 risk_score，否则使用 1 - pred_48
if "risk_score" in df.columns:
    risk = df["risk_score"].to_numpy()
else:
    risk = 1.0 - df["pred_48"].to_numpy()

# 生存概率矩阵 S(t)
surv_probs = eval_df[["pred_24", "pred_48", "pred_72"]].to_numpy()

# 为了 IPCW，需要提供训练分布，这里用全量数据近似
weighted_brier = calculate_weighted_brier(y, y, surv_probs, times=(24, 48, 72))

# C-index 使用原始数据，不包含伪样本
c_index = calculate_c_index(make_y(df["event"], df["time_to_hit_hours"]), risk)

mixed = calculate_mixed_score(c_index, weighted_brier)

print(f"C-index: {c_index:.4f}")
print(f"Weighted Brier: {weighted_brier:.4f}")
print(f"Mixed score: {mixed:.4f}")

C-index: 0.8356
Weighted Brier: 0.0814
Mixed score: 0.8937
